# Regularized Regression — Notebook 01 · Least Squares and Ridge

**Regularized Regression — From Likelihood to Lasso**

Covers **Task 1** (maximum likelihood by hand, and two ways for it to fail) and **Task 2** (Ridge as MAP with a Gaussian prior).

---

### How to run this

Put `uber_surge_mumbai.csv` (and, for notebook 02, `new_city_features.csv` and
`new_city_truth.csv`) in a `data/` folder beside this notebook, or upload them to your
Colab session. If the files are missing, the next cell regenerates them from the same
seed, so nothing here breaks either way.

In [ ]:
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True, linewidth=110)
pd.set_option('display.width', 120, 'display.max_columns', 30)

# --- a consistent look for every figure in this course -----------------------
plt.rcParams.update({
    'figure.figsize': (9, 4.6), 'figure.dpi': 110,
    'axes.grid': True, 'grid.alpha': .28, 'grid.linewidth': .7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c9cbc3', 'axes.labelcolor': '#4c514f',
    'axes.titlesize': 12, 'axes.titleweight': '600', 'axes.labelsize': 10,
    'xtick.color': '#767d7a', 'ytick.color': '#767d7a',
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.frameon': False, 'legend.fontsize': 9,
    'font.size': 10, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
})
C = {'ols': '#767d7a', 'ridge': '#2b5fa8', 'lasso': '#b4700c',
     'enet': '#5b4bb5', 'accent': '#1f6f5c', 'rose': '#b03a52'}

# Look in the usual places: beside the notebook, in ./data, or one level up
# (which is where the repo keeps them if you opened this from notebooks/).
SEARCH = ['data', '.', '../data', '..', '../../data']

def load(name):
    """Load a course CSV, regenerating it from the seed if it cannot be found."""
    for d in SEARCH:
        path = os.path.join(d, name)
        if os.path.exists(path):
            return pd.read_csv(path)
    for cand in ('scripts/make_data.py', 'make_data.py', '../scripts/make_data.py'):
        if os.path.exists(cand):
            subprocess.run([sys.executable, cand], check=True)
            return load(name)
    raise FileNotFoundError(
        f"{name} not found. Download it from the course site's Data page and put it "
        f"in a data/ folder beside this notebook.")

df = load('uber_surge_mumbai.csv')
print(f'{len(df)} trips, {df.shape[1]} columns')
df.head(3)

In [ ]:
# ---------------------------------------------------------------------------
# Conventions used throughout this course. Read these once; they save hours.
#
#   Objective (identical to sklearn's ElasticNet, including the 1/2n):
#       (1/2n)||y - Xw||^2  +  alpha*rho*||w||_1  +  (alpha*(1-rho)/2)*||w||^2
#
#   So the Ridge CLOSED FORM needs  n*alpha  where sklearn's Ridge takes alpha:
#       w = (X^T X + n*alpha*I)^-1 X^T y
#
#   Standardize on the TRAINING rows only, then apply that scaler to the test
#   rows. Split by time, never shuffled: trips 1-150 train, 151-200 test.
# ---------------------------------------------------------------------------
BASE8 = ['is_peak', 'is_rain', 'traffic_speed_kmph', 'drivers_available_500m',
         'is_event_nearby', 'is_airport_pickup', 'is_weekend', 'open_requests_500m']
SHORT = {'is_peak': 'is_peak', 'is_rain': 'is_rain',
         'traffic_speed_kmph': 'traffic_speed', 'drivers_available_500m': 'drivers_avail',
         'is_event_nearby': 'is_event', 'is_airport_pickup': 'is_airport',
         'is_weekend': 'is_weekend', 'open_requests_500m': 'open_requests',
         'is_bad_weather': 'is_bad_weather'}
N_TRAIN, SIGMA, TAU = 150, 10.0, 1.0

def split(frame, features):
    X = frame[features].to_numpy(float)
    y = frame['surge_additive_inr'].to_numpy(float)
    return X[:N_TRAIN], X[N_TRAIN:], y[:N_TRAIN], y[N_TRAIN:]

def standardize(Xtr, Xte):
    mu, sd = Xtr.mean(0), Xtr.std(0)
    sd = np.where(sd == 0, 1e-8, sd)
    return (Xtr - mu) / sd, (Xte - mu) / sd, mu, sd

def rmse(y, yhat):
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def r2(y, yhat):
    return float(1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2))

print('conventions loaded')

## Task 1a — An ill-conditioned pair

`is_rain` and `traffic_speed_kmph` on the **first six trips**. The two columns are
strongly correlated, and six rows leave barely enough independent variation to tell them
apart.

In [ ]:
A_FEATS = ['is_rain', 'traffic_speed_kmph']
n6 = 6

X = df[A_FEATS].to_numpy(float)[:n6]
y = df['surge_additive_inr'].to_numpy(float)[:n6]

# Standardize using ONLY these six rows: the estimator sees nothing else.
mu, sd = X.mean(0), X.std(0)
Z = (X - mu) / sd
yc = y - y.mean()

print('standardized design matrix Z:')
print(Z)
print('\ncentred target:', np.round(yc, 2))
print('\nsample correlation r =', round(float(np.corrcoef(X[:, 0], X[:, 1])[0, 1]), 4))

### Step 1 — build $X^\top X$ and $X^\top y$ by hand

Each entry of $X^\top X$ is a dot product of two columns. Since the columns are
standardized, the diagonal entries are exactly $n$ and the off-diagonal is $n\,r$, which
is why the correlation coefficient is the number that matters here.

In [ ]:
XtX = ...   # TODO
Xty = ...   # TODO
# TODO: your code here
raise NotImplementedError

### Step 2 — determinant and the explicit inverse

For a $2\times2$ matrix,
$\begin{pmatrix}a&b\\b&d\end{pmatrix}^{-1} = \frac{1}{ad-b^2}\begin{pmatrix}d&-b\\-b&a\end{pmatrix}$.

The inverse divides by the determinant, so everything the estimator produces —
coefficients and their standard errors — is scaled by $1/\det$.

In [ ]:
a, b, d = ...
det = ...
XtX_inv = ...
w6 = ...
# TODO: your code here
raise NotImplementedError

In [ ]:
# Prediction for trip 2 (19:00, peak and rain, 6 drivers)
pred2 = float(Z[1] @ w6 + y.mean())
print(f'predicted surge for trip 2: Rs {pred2:.2f}')
print(f'actually charged:           Rs {y[1]:.0f}')
print('\nClose — but the model saw six points, two of them at the price cap.')

### Step 3 — add trips one at a time

Refit on the first $n = 6, 7, \dots, 15$ trips and watch the coefficients. In a
well-posed problem, adding one observation out of six would nudge the answer.

In [ ]:
rows, prev = [], None
for n in range(6, 16):
    # TODO: refit on the first n trips, record r, det, both coefficients,
    #       and how far the estimate moved from the previous n.
    pass
# TODO: your code here
raise NotImplementedError

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(inst.n, inst.w_is_rain, 'o-', color=C['ridge'], label='is_rain')
ax[0].plot(inst.n, inst.w_traffic, 's-', color=C['rose'], label='traffic_speed')
ax[0].axhline(0, color='#c9cbc3', lw=1)
ax[0].set(xlabel='training trips n', ylabel='coefficient (Rs per s.d.)',
          title='Least squares, one trip at a time')
ax[0].legend()
ax[1].plot(inst.n, inst.det, 'o-', color=C['accent'])
ax[1].set(xlabel='training trips n', ylabel='det(X$^T$X)',
          title='The determinant recovering as data arrives')
plt.tight_layout(); plt.show()

> **Write-up prompt (Task 1a).** Reporting that the coefficients move is not enough.
> Explain why, in terms of the determinant and the $-r$ off-diagonal of the inverse. One
> useful extra check: is the **sum** of the two coefficients more stable than either
> alone? If so, what does that say about which quantity the data identifies?

## Task 1b — A pair where OLS does not exist

Now `is_rain` and `is_bad_weather` on the **first ten trips**.

In [ ]:
df['is_bad_weather'] = ((df.is_rain == 1) & (df.traffic_speed_kmph < 25)).astype(int)

sub = df[['is_rain', 'traffic_speed_kmph', 'is_bad_weather']].head(10)
print(sub.to_string())
print('\ncolumns identical on these rows:',
      bool((sub.is_rain == sub.is_bad_weather).all()))

In [ ]:
# TODO: build XtX on the first ten trips, report its determinant,
#       try to invert it, and find the smallest n at which it becomes solvable.
# TODO: your code here
raise NotImplementedError

> **Write-up prompt (Task 1b, step 3).** "The MLE does not exist" is a claim about the
> likelihood surface, not about NumPy. Adding $c$ to one coefficient and subtracting $c$
> from the other changes no prediction, so the likelihood has a flat direction and no
> unique maximum. Say that in one sentence, in your own words.

## Task 1c — The real thing: eight features, 150 trips

In [ ]:
Xtr, Xte, ytr, yte = split(df, BASE8)
Ztr, Zte, mu8, sd8 = standardize(Xtr, Xte)
ybar = ytr.mean()
w_ols = ...   # TODO: solve the normal equations
# TODO: your code here
raise NotImplementedError

> **Note the condition number.** An order of magnitude better than the six-row problem.
> With 150 trips and 8 features this dataset is not ill-conditioned, which is worth
> remembering when cross-validation picks a tiny penalty in Task 2.

---

## Task 2 — Ridge regression as MAP

Gaussian prior $w \sim \mathcal{N}(0, \tau^2)$ gives
$\hat w = (X^\top X + \lambda I)^{-1}X^\top y$ with $\lambda = \sigma^2/\tau^2$.

Use `np.linalg.solve`, not `np.linalg.inv`: the explicit inverse is slower and
numerically worse. Task 1 formed one only because we wanted to look at it.

In [ ]:
def ridge_fit(Z, y, alpha_sklearn):
    """Closed-form ridge. Use np.linalg.solve."""
    # TODO: your code here
    raise NotImplementedError

yc_tr = ytr - ybar
# TODO: loop over alpha in [0.1, 1, 10, 100, 1000] and build the table

In [ ]:
# Verify against scikit-learn (allowed here: we are CHECKING, not computing)
from sklearn.linear_model import Ridge
for a in grid:
    mine = ridge_fit(Ztr, yc_tr, a)
    theirs = Ridge(alpha=a, fit_intercept=False).fit(Ztr, yc_tr).coef_
    print(f'alpha={a:>6}:  max |difference| = {np.max(np.abs(mine - theirs)):.2e}')

### The coefficient path (Deliverable 1, panel 1)

In [ ]:
alphas = np.logspace(-2, 4, 80)
paths = np.array([ridge_fit(Ztr, yc_tr, a) for a in alphas])

fig, ax = plt.subplots(figsize=(10, 5))
for j, f in enumerate(BASE8):
    ax.plot(alphas, paths[:, j], lw=1.9, label=SHORT[f])
ax.set_xscale('log')
ax.axhline(0, color='#c9cbc3', lw=1)
ax.set(xlabel=r'penalty $\alpha$  (log scale)', ylabel='coefficient (Rs per s.d.)',
       title='Ridge path — everything shrinks, nothing reaches zero')
ax.legend(ncol=2, loc='upper right')
plt.tight_layout(); plt.show()

gaps = np.abs(paths[:, 0] - paths[:, 1])
print('gap between is_peak and is_rain:')
for a, g in zip(alphas[::16], gaps[::16]):
    print(f'  alpha={a:9.2f}   |gap| = {g:5.2f}')
print('\nThe gap falls monotonically: the two correlated features converge toward')
print('each other rather than meeting at some particular alpha.')

### Choosing $\alpha$ honestly

Cross-validate **on the training split only**. The test set is for reporting, once.

In [ ]:
def kfold_cv(Z, y, k, fit):
    """k-fold CV RMSE. Re-centre y inside each fold."""
    # TODO: your code here
    raise NotImplementedError

# TODO: sweep alpha, record train / CV / test RMSE, plot all three, report the minimum

> **Write-up prompt (Task 2, step 6).** Cross-validation and the stated prior disagree by
> a wide margin. The prior $\tau = 1$ asserts coefficients of about $\pm1$ rupee per
> standard deviation; the data says about $\pm10$. Which do you trust with 150
> observations, and what does the disagreement say about the prior?

---

### Checklist before you move on

- [ ] `XtX` computed by hand and cross-checked against NumPy
- [ ] Instability table produced, with the largest one-trip move quoted
- [ ] Singular case reproduced, and explained as a property of the likelihood
- [ ] Ridge implemented with `solve`, matching sklearn to ~1e-14
- [ ] Coefficient path plotted with a log x-axis (**Deliverable 1**)
- [ ] CV run on the training split only, and the CV-vs-prior disagreement discussed

Next: **Notebook 02 — Lasso by coordinate descent**.